# Train SASRec

Trains a SASRec sequential recommender from a `.inter` atomic file.

In [1]:
import numpy as np

# For NumPy 2.0 compatibility with RecBole 1.2
np.float_ = np.float64
np.int_ = np.int64
np.complex_ = np.complex128
np.unicode_ = np.str_

In [2]:
from typing import Any
import torch
from pathlib import Path
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.model.sequential_recommender import SASRec
from recbole.trainer import Trainer
from recbole.utils import init_seed

In [3]:
# --- Config ---
DATASET_NAME: str = "Beauty_and_Personal_Care"
DATA_DIR: str = "../data"
SEED = 67

### Create dataset

RecBole's `SequentialDataset` automatically builds item sequences per user (ordered by timestamp).

In [4]:
init_seed(SEED, reproducibility=True)

In [5]:
config_dict: dict[str, Any] = {
    "data_path": DATA_DIR,
    "dataset": DATASET_NAME,
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "RATING_FIELD": "rating",
    "TIME_FIELD": "timestamp",
    "load_col": {
        "inter": ["user_id", "item_id", "rating", "timestamp"],
    },
    "MAX_ITEM_LIST_LENGTH": 50,
    "epochs": 1,
    "train_batch_size": 512,
    "eval_batch_size": 512,
    "train_neg_sample_args": None,
    "eval_args": {
        "split": {"RS": [8, 1, 1]},
        "group_by": "user",
        "order": "TO",
        "mode": "full",
    },
    "metrics": ["NDCG", "Recall", "MRR"],
    "valid_metric": "NDCG@10",
    "seed": SEED,
}

config: Config = Config(model="SASRec", config_dict=config_dict)

# Use MPS if available (for Apple Silicon)
if torch.backends.mps.is_available():
    print("Configuring MPS (Apple Silicon GPU) for training.")
    config.final_config_dict["device"] = torch.device("mps")

dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)
print(f"Users: {dataset.user_num:,}  Items: {dataset.item_num:,}")
print(f"Train interactions: {len(train_data.dataset):,}")

Configuring MPS (Apple Silicon GPU) for training.


/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/data/dataset/dataset.py:648: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  feat[field].fillna(value=0, inplace=True)
/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/data/dataset/dataset.py:650: ChainedAssignme

Users: 330,738  Items: 209,390
Train interactions: 477,823


### Train SASRec

In [6]:
model: SASRec = SASRec(config, train_data.dataset).to(config["device"])
trainer: Trainer = Trainer(config, model)

best_valid_score: float
best_valid_result: dict[str, float]
best_valid_score, best_valid_result = trainer.fit(train_data, valid_data, verbose=True, show_progress=True)

Train     0:   0%|                                                          | 0/934 [00:00<?, ?it/s]/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/trainer/trainer.py:235: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler(enabled=self.enable_scaler)
Evaluate   : 100%|████████████████████████████████████████████████| 155/155 [00:14<00:00, 10.56it/s]


In [7]:
print(f"\nBest valid score: {best_valid_score:.4f}")
print("Best valid result:")
for metric, score in best_valid_result.items():
    print(f"  {metric}: {score:.4f}")


Best valid score: 0.0062
Best valid result:
  ndcg@10: 0.0062
  recall@10: 0.0118
  mrr@10: 0.0045


### Evaluate on test set

In [8]:
test_result: dict[str, float] = trainer.evaluate(test_data)

print("Test results:")
for metric, value in test_result.items():
    print(f"  {metric}: {value:.4f}")

/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/trainer/trainer.py:583: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = tor

Test results:
  ndcg@10: 0.0071
  recall@10: 0.0132
  mrr@10: 0.0053
